<a href="https://colab.research.google.com/github/reza-nz/image-captioning-project/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# GENERAL IMPORTS
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

## A – Dataset Preparation

### Step A.0 – Download the dataset 

In [17]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub

adityajn105_flickr8k_path = kagglehub.dataset_download('adityajn105/flickr8k')

print('Data source import complete.')


Data source import complete.


### Step A.1 Inspect the downloaded files

In [18]:
DATA_DIR = Path(adityajn105_flickr8k_path)

# Confirm that the dataset was imported correctly by printing the path and listing the files/folders inside it
print(DATA_DIR)
print("Files/folders inside dataset:")
for path in DATA_DIR.iterdir():
    print(path.name)

# Define paths to the images and captions file
IMAGES_DIR = DATA_DIR / "Images"
CAPTIONS_FILE = DATA_DIR / "captions.txt"

/home/codespace/.cache/kagglehub/datasets/adityajn105/flickr8k/versions/1
Files/folders inside dataset:
Images
captions.txt


### Step A.2 Read and verify the files

We first inspect the Flickr8k dataset structure and verify that captions.txt contains 40,455 captions for 8,091 images, with exactly five captions per image. 

Furthermore we also check that every image referenced in the captions file exists in the Images folder and that there are no unused images, so the dataset is complete and ready for caption preprocessing.

In [19]:
# Verify the captions table 
print("======== Captions File Check =======")
captions_df = pd.read_csv(CAPTIONS_FILE)
print(captions_df.head())
print()
print(captions_df.info())


# Sanity check the captions data
print("======== Captions Data Sanity Check =======")
print("Number of caption rows:", len(captions_df))
print("Number of unique images in captions:", captions_df["image"].nunique())

captions_per_image = captions_df.groupby("image")["caption"].count()

print("Minimum captions per image:", captions_per_image.min())
print("Maximum captions per image:", captions_per_image.max())
print("Average captions per image:", captions_per_image.mean())

======== Captions File Check =======
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  

<class 'pandas.DataFrame'>
RangeIndex: 40455 entries, 0 to 40454
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   image    40455 non-null  str  
 1   caption  40455 non-null  str  
dtypes: str(2)
memory usage: 632.2 KB
None
======== Captions Data Sanity Check =======
Number of caption rows: 40455
Number of unique images in captions: 8091
Minimum captions per image: 5
Maximum c

In [20]:
# Check for missing or unused images
image_files = set(path.name for path in IMAGES_DIR.glob("*.jpg"))
caption_image_files = set(captions_df["image"].unique())

missing_images = caption_image_files - image_files
unused_images = image_files - caption_image_files

print("Images in folder:", len(image_files))
print("Images referenced in captions:", len(caption_image_files))
print("Missing images:", len(missing_images))
print("Unused images:", len(unused_images))

Images in folder: 8091
Images referenced in captions: 8091
Missing images: 0
Unused images: 0


### Step A.3 Implement image preprocessing

Here we implement and test the image preprocessing pipeline.
Each image is loaded in RGB format, resized and centre-cropped to 224 × 224 pixels, converted into a PyTorch tensor, and normalised with ImageNet statistics so that it is compatible with pretrained CNN encoders (in our case ResNet).
Last but not least a sanity check: it confirmed that all dataset images can be successfully transformed into tensors of shape [3, 224, 224].


#### Here some worthwhile comments 

THOUGHT: it might be worth checking if `transforms.Resize((224, 224))` is (also or more) appropriate for our dataset.

Explaination on the mean and std values I used: (https://stackoverflow.com/questions/58151507/why-pytorch-officially-use-mean-0-485-0-456-0-406-and-std-0-229-0-224-0-2)

Useful quote: »Whether or not to use ImageNet's mean and stddev depends on your data. Assuming your data are ordinary photos of "natural scenes"† (people, buildings, animals, varied lighting/angles/backgrounds, etc.), and assuming your dataset is biased in the same way ImageNet is (in terms of class balance), then it's ok to normalize with ImageNet's scene statistics« 

In [21]:
image_transform = transforms.Compose([
    transforms.Resize(256),  # Resize the shorter side to 256 pixels while maintaining aspect ratio
    transforms.CenterCrop(224),  # Crop the center 224x224 pixels to match the input size expected by ResNet
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [23]:
def check_images_load(image_names, images_dir, transform):
    """
    Checks whether all images can be opened and transformed.
    Returns a list of problematic image filenames.
    """
    broken_images = []

    for image_name in tqdm(image_names):
        image_path = images_dir / image_name

        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
                image_tensor = transform(image)

            if image_tensor.shape != (3, 224, 224):
                broken_images.append(image_name)

        except Exception as error:
            print(f"Problem with {image_name}: {error}")
            broken_images.append(image_name)

    return broken_images

In [24]:
unique_image_names = captions_df["image"].unique()

broken_images = check_images_load(
    image_names=unique_image_names,
    images_dir=IMAGES_DIR,
    transform=image_transform
)

print("Number of problematic images:", len(broken_images))

100%|██████████| 8091/8091 [00:57<00:00, 141.28it/s]

Number of problematic images: 0


# STICKY NOTE: The image preprocessing design and validation are complete. The remaining step is to integrate this transform into the custom PyTorch Dataset.
# and the captions still need to be preprocessed as well (e.g. tokenization etc.)